In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px
from sklearn.metrics import confusion_matrix, multilabel_confusion_matrix
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression

## Loading Datatset

In [2]:
df = pd.read_csv('../dataset/students_data.csv')

In [3]:
df.head()

,age,num_notebooks,num_friends,travel_time,chai_cups_daily,total_f_grades,hangouts,societies,TAships,health,absences,times_cheated,study_time,love_for_ai/ml,grade
0,21,4,2,1,3,3,4,4,5,3,14,8,7,2,E
1,20,2,3,4,3,1,5,1,1,3,16,15,12,11,B
2,21,0,0,4,3,0,4,2,2,5,16,11,10,8,C
3,17,0,1,3,3,1,3,4,4,2,13,10,10,6,E
4,19,1,0,4,3,0,3,2,2,4,4,12,14,12,B


## Train Test Split

In [4]:
X = df.drop(columns=['grade'], axis=1)
y = df['grade']

In [5]:
X_train, X_test, y_train, y_test =  train_test_split(
    X,
    y,
    random_state=42,
    stratify=y
)

## Feature Scaling

In [6]:
scaler =  StandardScaler()

In [7]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Visualization of Classes

## 2D

In [8]:
pca = PCA(n_components=2)
x_pca = pca.fit_transform(X)
pca_df_2d = pd.DataFrame(
    x_pca,
    columns=['C1', 'C2']
)
pca_df_2d['class'] = y

pca_fig_2d = px.scatter(
    pca_df_2d,
    x='C1',
    y='C2',
    color='class',
    title='2D PCA Projection'
)
pca_fig_2d.show()
pca_fig_2d.write_image("../visualizations/multi_class/2d_pca_multiclass_distribution.png")

In [16]:
pca = PCA(n_components=3)
x_pca = pca.fit_transform(X)
pca_df = pd.DataFrame(
    x_pca,
    columns=['C1', 'C2', 'C3']
)
pca_df['class'] = y

pca_fig = px.scatter_3d(
    pca_df,
    x='C1',
    y='C2',
    z='C3',
    color='class',
    title='PCA Projection'
)
pca_fig.show()
pca_fig.write_image("../visualizations/multi_class/3d_pca_multiclass_distribution.png")

## Implementation

In [9]:
class LogisticRegression:

    def __init__(self, lr=0.01, iterations=1000, lambda_val=0.1):
        self.lr = lr
        self.iterations = iterations
        self.lambda_val = lambda_val
        self.loss_history = []

    def binary_cross_entropy(self, y, y_hat):
        eps = 1e-8
        return -(y*torch.log(y_hat + eps) + (1-y)*(torch.log(1-y_hat+eps))).mean()

    def fit(self, X, y):
        m, n = X.shape
        self.weights = torch.zeros(n, dtype=torch.float32, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float32, requires_grad=True)

        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        for epoch in range(self.iterations):
            z = X @ self.weights + self.bias
            y_hat = torch.sigmoid(z)

            bce = self.binary_cross_entropy(y, y_hat)

            l2_penalty = 0
            # Uncomment to apply regularization
            # l2_penalty = (self.lambda_val/2*m) * torch.sum(self.weights ** 2)

            loss = bce + l2_penalty
            self.loss_history.append(loss.item())
            loss.backward()

            with torch.no_grad():
                self.weights -= self.lr * self.weights.grad
                self.bias -= self.lr * self.bias.grad

            self.weights.grad.zero_()
            self.bias.grad.zero_()

    def predict(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        z = X @ self.weights + self.bias
        y_hat = torch.sigmoid(z)

        return (y_hat>=0.5).numpy()


### Using One-vs-Rest Strategy

In [10]:
def get_model_results(X, y):
    classifiers = {}
    classes = np.unique(y)

    for cur_class in classes:
        label = (y==cur_class).astype(int)
        model = LogisticRegression()
        model.fit(X, label)
        classifiers[cur_class] = model

    return classifiers

In [11]:
classifiers = get_model_results(X_train, y_train)

In [12]:
for cur_class, model in classifiers.items():
    
    preds = model.predict(X_test)
    true_label = (y_test==cur_class).astype(int)
    cm = confusion_matrix(true_label, preds)
    print("="*10, '\n', cm, '\n', "="*10, "\n")

 [[99  3]
 [ 7 16]] 

 [[80 12]
 [32  1]] 

 [[83  6]
 [36  0]] 

 [[95  9]
 [21  0]] 

 [[113   0]
 [  3   9]] 



## Loss Curves

In [13]:
for cur_class, model in classifiers.items():
    fig = px.line(
        model.loss_history
    )
    fig.show()

## Visualizing Boundaries

In [14]:
classifiers = get_model_results(pca_df_2d[['C1', 'C2']].to_numpy(dtype=np.float32), y)

In [15]:
for cur_class, model in classifiers.items():
    w = model.weights.detach().numpy()
    b = model.bias.detach().numpy()

    x_line = np.linspace(
        pca_df_2d['C1'].min(),
        pca_df_2d['C2'].max(),
        300
    )

    y_line = -(w[0] * x_line + b) / w[1]
    mask = (
        (y_line >= pca_df_2d['C2'].min()) &
        (y_line <= pca_df_2d['C2'].max())
    )
    pca_fig_2d.add_scatter(
        x = x_line[mask],
        y = y_line[mask],
        mode = 'lines',
        line=dict(color="black", width=3),
        name='Decision Boundary' 
    )
    pca_fig_2d.show()
    pca_fig_2d.write_image(f"../visualizations/multi_class/{cur_class}_boundary_in_2d_pca.png")

## Sklearn Implementation

In [29]:
sklearn_model = SklearnLogisticRegression()

In [30]:
sklearn_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [32]:
sklearn_preds = sklearn_model.predict(X_test)

In [33]:
cm = confusion_matrix(y_test,sklearn_preds)

In [35]:
print(cm)

[[18  5  0  0  0]
 [ 2 25  6  0  0]
 [ 0  7 26  3  0]
 [ 0  0  6 12  3]
 [ 0  0  0  2 10]]


In [37]:
cms = multilabel_confusion_matrix(y_test, sklearn_preds)

classes = sklearn_model.classes_

for cls, cm in zip(classes, cms):
    print(f"Confusion Matrix for class {cls}")
    print(cm)
    print()

Confusion Matrix for class A
[[100   2]
 [  5  18]]

Confusion Matrix for class B
[[80 12]
 [ 8 25]]

Confusion Matrix for class C
[[77 12]
 [10 26]]

Confusion Matrix for class D
[[99  5]
 [ 9 12]]

Confusion Matrix for class E
[[110   3]
 [  2  10]]

